# Getting started

This notebook runs one experiment five ways, on one synthetic cohort: with a preset and the
pipeline, as the plan the preset resolves to, as the public verbs one cell at a time, with one
step changed, and back through the pipeline with that change. Every run ends on soma's own
nouns: the run directory, `Predictions`, `Evaluation` and `compare`.

It needs no GPU, no model weights and no network: the slides are written by
`soma.testing.synthetic_cohort` and embedded by `fake`, a weight-free encoder. On a laptop CPU it
runs in a few minutes. [Install soma](installation.rst) first.

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = ""  # run on the CPU, so the numbers are the same on every machine

import sys
import tempfile
from dataclasses import replace
from pathlib import Path

import pandas as pd
import yaml

import soma
from soma.config import Component

root = Path(tempfile.mkdtemp(prefix="soma-getting-started-"))

## 1. Run an experiment

`soma.testing.synthetic_cohort` writes 32 small whole-slide images and the two tables soma reads.
Your data takes the same shape: `cohort.csv` holds one row per slide (`sample_id`, `image_path`,
the target `label`, an optional `patient_id` and any metadata column), and `splits.csv` gives
each slide its split.

In [ ]:
records, splits = soma.testing.synthetic_cohort(root / "data", n=32, seed=0)
pd.read_csv(records).head()

In [ ]:
pd.read_csv(splits).head()

Each slide holds a tissue disc whose colour darkens with `severity`, and `label` is
`severity > 0.5`: a model scores well only if the right pixels, features and labels reach it.
Two slides of one patient always share a split.

Every swappable part of soma is a component of some kind. `soma.components()` lists the kinds,
and `soma.components(kind)` the registered entries of one kind, with their params:

In [ ]:
soma.components()

In [ ]:
[entry["name"] for entry in soma.components("aggregators")]

A preset returns a complete experiment, an `ExperimentConfig`, from a few arguments.
`mil_classification` tiles each slide, embeds the tiles with a frozen encoder, pools each slide's
bag of tile features with the MIL aggregator you name (AB-MIL here) and classifies it.

The config is a dataclass: change it by assigning fields. The function below points a preset's
config at the cohort and makes the run fit this toy cohort: all slides are tiled and loaded in
this process, the tile features are z-scored on the train split, and the model trains 20 epochs at
a higher learning rate. Every run below goes through it.

In [ ]:
def on_this_cohort(cfg):
    cfg.cohort.records, cfg.cohort.splits = records, splits
    cfg.run.output_dir = root / "runs"
    cfg.features.cache.root_dir = root / "cache"  # one feature cache for every run below
    cfg.features.execution = replace(cfg.features.execution, num_workers_per_gpu=0, num_preprocessing_workers=0)
    cfg.learner.params["normalization"] = {"method": "zscore"}
    cfg.learner.params["optimizer"] = Component("adamw", {"lr": 3e-3})
    cfg.trainer.max_epochs = 20
    return cfg


cfg = on_this_cohort(soma.presets.mil_classification(encoder="fake", aggregator="abmil"))
result = soma.Pipeline(cfg).run()
result.summary

The run directory holds everything the run decided and produced: `config.yaml` (the config as
given), `plan.yaml` (the config with every default filled in), the test predictions, the
metrics, the training history and the trained model as a `bundle/` that `soma.load` reopens.

In [ ]:
sorted(path.name for path in result.run_dir.iterdir())

The predictions are a file of their own. `soma.evaluate` scores them against the cohort's
targets with any metric list, without the model:

In [ ]:
cohort = soma.Cohort.from_csv(records, splits, targets=["label"])
from_run = soma.Predictions.from_parquet(result.run_dir / "predictions_test.parquet")
soma.evaluate(from_run, cohort.targets("test"), metrics=["accuracy", "mcc"]).metrics

Swapping a component is one argument. The same experiment with mean pooling in place of AB-MIL
(the tile features come from the cache this time, nothing is encoded again):

In [ ]:
mean_pool = soma.Pipeline(on_this_cohort(soma.presets.mil_classification(encoder="fake", aggregator="mean_pool"))).run()
mean_pool.summary

On the command line, `soma init` prints a preset as an editable YAML file with every default
spelled out. Its first sections:

In [ ]:
!{sys.executable} -m soma init mil_classification --encoder fake --aggregator abmil > {root}/experiment.yaml
text = (root / "experiment.yaml").read_text()
print(text[: text.index("features:")] + "...")

The file and the Python object are one config: with the same edits, the file resolves to the same
plan as `cfg`.

In [ ]:
from_file = on_this_cohort(soma.ExperimentConfig.from_yaml(root / "experiment.yaml"))
soma.resolve(from_file, cohort) == soma.resolve(cfg, cohort)

After the same edits in the file, `soma run experiment.yaml` runs this experiment. A preset also
runs as is in one line:

```bash
soma run --preset mil_classification --encoder fake --aggregator abmil --cohort cohort.csv --splits splits.csv --out runs
```

On your own slides, name a foundation model instead of `fake` and point the cohort at your
tables; nothing else changes:

```python
cfg = soma.presets.mil_classification(encoder="uni2", aggregator="abmil")
cfg.cohort.records, cfg.cohort.splits = "cohort.csv", "splits.csv"  # sample_id, image_path, label[, patient_id]
cfg.run.output_dir = "runs"
soma.Pipeline(cfg).run()
```

The first run downloads the encoder's weights (`uni2` is gated on Hugging Face: accept its terms
and set `HF_TOKEN`); feature extraction is much faster on a GPU, and is cached for every later run.

## 2. What the preset resolved to

`soma.resolve` expands a config against its cohort into a `Plan`: every `{name, params}` leaf
with all its params, and what the cohort decides (here the class count and the label map)
filled in. The run wrote the same plan to `plan.yaml` before it trained.

In [ ]:
plan = soma.resolve(cfg, cohort)
plan.shape, plan.task_family, plan.unit

In [ ]:
resolved = plan.to_dict()
print(yaml.safe_dump(
    {"features.encoder": resolved["features"]["encoder"], "learner": resolved["learner"], "trainer": resolved["trainer"]},
    sort_keys=False,
))

In [ ]:
soma.Plan.from_yaml(result.run_dir / "plan.yaml").comparable() == plan.comparable()

## 3. The same run from the verbs

`Pipeline` composes the public verbs in one fixed order. Below, each step is one cell, built with
plain constructors from the plan's values; the result is the pipeline's run, byte for byte.

**Seed.** Seeds Python, NumPy and torch for fold 0: every random draw below follows from it.

In [ ]:
soma.seed_everything(plan.run.seed, fold=0)

**Source.** `soma.extract` tiles the slides and embeds the tiles, once: the cache serves the
pipeline's features again, so nothing is encoded here. This is where another encoder goes.

In [ ]:
source = soma.extract(
    soma.ImageManifest.from_csv(records),
    encoder="fake",
    extraction=cfg.features.extraction,
    execution=cfg.features.execution,
    cache=cfg.features.cache,
    output_dir=root / "by-hand",
)
source.feature_dim, source.load("slide00").shape

Features you computed yourself enter through a source too, for example `soma.data.from_arrays`
over one `(tiles, features)` tensor per slide; `soma.testing.check_set_source` checks any source:

In [ ]:
own = soma.data.from_arrays({sample_id: source.load(sample_id) for sample_id in source.sample_ids})
soma.testing.check_set_source(own)

**Cohort.** The cohort holds the targets and the folds, and refuses a patient whose slides fall in two splits.
The other cohorts you would build here: patient-level bags (`cohort.collapse("patient_id")`) or a
cross-validation `splits.csv` with a `fold` column.

In [ ]:
fold = cohort.folds[0]
len(fold.train), len(fold.tune), len(fold.tests["test"])

**Feature adaptor.** The z-score statistics are fitted on the train split only, so no tune or test
slide leaks into them. A projection (PCA, random) would be fitted here as well.

In [ ]:
from soma.learn.feature_adaptor import FeatureAdaptor

adaptor = FeatureAdaptor(method="zscore", num_features=source.feature_dim)
adaptor.fit([source.load(sample_id) for sample_id in fold.train])

**Model.** The aggregator, then the head, behind the adaptor; `record_builder` notes how to
rebuild the model, so that `soma.load` can reopen it later. The learner owns the optimization.
This is where another aggregator, head or optimizer goes.

In [ ]:
def build_model(feature_dim, adaptor=None):
    aggregator = soma.models.ABMIL(input_dim=feature_dim)
    head = soma.models.MulticlassClassificationHead(input_dim=aggregator.output_dim, num_classes=2, label_map={0: 0, 1: 1})
    adaptor = adaptor or FeatureAdaptor(method="zscore", num_features=feature_dim)
    model = soma.models.MILModel(aggregator, head, feature_adaptor=adaptor)
    return soma.learn.record_builder(model, build_model, feature_dim=feature_dim)


learner = soma.learn.MILLearner(
    build_model(source.feature_dim, adaptor), optimizer={"name": "adamw", "params": {"lr": 3e-3}}, schedulers=["cosine"]
)

**Loaders.** One bag per batch, shuffled for training. A sampler or an augmentation goes here.

In [ ]:
loaders = soma.learn.loaders(source, cohort, head=learner.head, batch_size=1)

**Baseline.** Slides where tiling finds no tissue have no features; they are scored with the
train split's class prior so that every test slide is counted. Here every slide has tissue.

In [ ]:
head = learner.head
baseline = soma.evaluation.Baseline.fit(
    cohort.targets("train"), task_family=head.task_family, classes=head.classes, target_keys=head.target_keys
)
no_features = [sample_id for sample_id in fold.tests["test"] if sample_id not in source.sample_ids]

**Fit.** Trains with the plan's callbacks: keep the epoch with the best tune loss, stop after 10
epochs without improvement.

In [ ]:
fit = soma.fit(learner, loaders.train, loaders.tune, max_epochs=20)

**Tune step.** Classification has none. Detection chooses its peak threshold on the tune
predictions here (`soma.evaluation.sweep_threshold`).

**Predict and evaluate.** The test predictions, plus baseline rows for the slides without
features, scored against the test targets:

In [ ]:
by_hand = soma.Predictions.concat([soma.predict(fit.learner, loaders.tests["test"]), baseline.predict(no_features)])
soma.evaluate(by_hand, cohort.targets("test")).metrics

**Save.** The bundle holds the weights and the recipe that rebuilds the model; `soma.load`
reopens it, and the reopened model scores the same:

In [ ]:
bundle = soma.save(fit, root / "by-hand" / "bundle")
reopened = soma.load(bundle)
soma.evaluate(soma.predict(reopened.model, loaders.tests["test"]), cohort.targets("test")).metrics

The hand-built run is the pipeline's run: the same predictions (`compare`'s paired permutation
test finds no difference) and the same weights. soma's test suite holds every pipeline run to
this.

In [ ]:
(
    soma.evaluation.compare([from_run, by_hand], cohort.targets("test"), "auroc_macro"),
    from_run.frame.equals(by_hand.frame),
    (result.run_dir / "bundle" / "weights.safetensors").read_bytes() == (bundle / "weights.safetensors").read_bytes(),
)

## 4. Change a step

A learner is a class whose methods are the steps of training. This one adds a penalty on the
entropy of the attention, so each slide's prediction rests on fewer tiles: one
`training_step` override.

In [ ]:
class EntropyMIL(soma.learn.MILLearner):
    """MIL plus 0.01 x the entropy of each bag's attention over its tiles."""

    def training_step(self, batch):
        output = self.forward_batch(batch)
        attention = output.tile_attention.masked_fill(~batch.mask, float("-inf")).softmax(-1)
        entropy = -(attention * attention.clamp_min(1e-8).log()).sum(-1).mean()
        return self.compute_loss(output, batch) + 0.01 * entropy

`soma.testing.check_learner` runs one batch through every method soma calls and names the one
that breaks the contract, if any. The batch here is the eight tune slides, so that it holds both
classes:

In [ ]:
probe = EntropyMIL(build_model(source.feature_dim, adaptor))
batch = next(iter(soma.learn.loaders(source, cohort, head=probe.head, batch_size=8).tune))
soma.testing.check_learner(probe, batch, budget=soma.learn.Budget(max_epochs=1, max_steps=None, updates_per_epoch=1))

The same cells as above, with `EntropyMIL` in place of `MILLearner`:

In [ ]:
soma.seed_everything(plan.run.seed, fold=0)
adaptor = FeatureAdaptor(method="zscore", num_features=source.feature_dim)
adaptor.fit([source.load(sample_id) for sample_id in fold.train])
learner = EntropyMIL(
    build_model(source.feature_dim, adaptor), optimizer={"name": "adamw", "params": {"lr": 3e-3}}, schedulers=["cosine"]
)
loaders = soma.learn.loaders(source, cohort, head=learner.head, batch_size=1)
fit = soma.fit(learner, loaders.train, loaders.tune, max_epochs=20)
entropy_by_hand = soma.Predictions.concat([soma.predict(fit.learner, loaders.tests["test"]), baseline.predict(no_features)])
soma.evaluate(entropy_by_hand, cohort.targets("test")).metrics

## 5. Hand it back to the pipeline

A config names a class the registry does not know by its dotted import path: a class defined in
this notebook is `__main__.EntropyMIL`. soma warns that it cannot validate the class's params,
then runs it like any other learner, and the run is again the hand-built one:

In [ ]:
cfg.learner = replace(cfg.learner, name="__main__.EntropyMIL")
entropy_run = soma.Pipeline(cfg).run()
from_entropy_run = soma.Predictions.from_parquet(entropy_run.run_dir / "predictions_test.parquet")
soma.evaluation.compare([from_entropy_run, entropy_by_hand], cohort.targets("test"), "auroc_macro"), from_entropy_run.frame.equals(entropy_by_hand.frame)

To share the learner, put it in a package and register it, so its params are checked and it is
listed by `soma.components("learners")`: see [Extending soma](extending.rst) for this and the other
extension points, each with its conformance check.